# Recommender Systems — Python Examples

**Completed / instructor version**

This notebook accompanies the introductory lecture on recommender systems.

We will implement:

1. **Content-based filtering**
2. **User-based collaborative filtering**
3. **Item-based collaborative filtering**
4. **Precision@K and Recall@K**
5. **Average Precision**

The goal is not to build a production recommender. The goal is to make the lecture concepts concrete and transparent.

A recurring question throughout the notebook is:

> **What exactly are we comparing?**

## 0. Setup

We use:

- `pandas` for tables
- `numpy` for numerical operations
- `scikit-learn` for cosine similarity in the **content-based** example

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.precision", 3)

# 1. Content-based filtering

A content-based recommender compares **item features** with a **user profile**.

Each movie below is described by simple binary genre features: `1` means the feature is present, `0` means it is not.

In [2]:
movies = pd.DataFrame({
    "Sci-Fi":  [1, 1, 1, 1, 0, 0, 0],
    "Action":  [1, 0, 0, 1, 1, 0, 0],
    "Drama":   [0, 1, 1, 1, 0, 1, 0],
    "Romance": [0, 0, 0, 0, 0, 0, 1],
    "Comedy":  [0, 0, 0, 0, 0, 1, 1]
}, index=[
    "Matrix",
    "Dune",
    "Arrival",
    "Blade Runner 2049",
    "John Wick",
    "Barbie",
    "Mamma Mia"
])

movies

,Sci-Fi,Action,Drama,Romance,Comedy
Matrix,1,1,0,0,0
Dune,1,0,1,0,0
Arrival,1,0,1,0,0
Blade Runner 2049,1,1,1,0,0
John Wick,0,1,0,0,0
Barbie,0,0,1,0,1
Mamma Mia,0,0,0,1,1


Karen likes **Matrix** and **Dune**.

A simple user profile can be created by averaging the features of the movies she likes.

In [3]:
liked_movies = ["Matrix", "Dune"]

user_profile = movies.loc[liked_movies].mean(axis=0)

user_profile

Sci-Fi     1.0
Action     0.5
Drama      0.5
Romance    0.0
Comedy     0.0
dtype: float64

Interpretation:

- Sci-Fi = 1.0 → strong evidence that Karen likes Sci-Fi
- Action = 0.5
- Drama = 0.5
- Romance / Comedy = 0.0 → **no evidence yet**, not necessarily a dislike

For this type of feature data, we use **cosine similarity** to compare Karen's profile with the candidate movies.

In [4]:
scores = cosine_similarity(
    user_profile.values.reshape(1, -1),
    movies.values
).flatten()

recommendations = pd.Series(scores, index=movies.index)

# Do not recommend movies Karen already watched.
recommendations = recommendations.drop(liked_movies)

recommendations.sort_values(ascending=False)

Blade Runner 2049    0.943
Arrival              0.866
John Wick            0.408
Barbie               0.289
Mamma Mia            0.000
dtype: float64

### Key idea

Content-based filtering recommends items because their **features are similar** to the features of items the user already likes.

It does **not** need other users' ratings.

# 2. Collaborative filtering data

Collaborative filtering uses **user-item interactions** instead of item features.

The ratings matrix below matches the lecture example. `NaN` means **unknown / not rated**.

> Missing rating ≠ rating of 0

In [5]:
ratings = pd.DataFrame({
    "Matrix":     [5, 5, 1, 4],
    "Dune":       [5, 4, 1, 5],
    "Barbie":     [np.nan, 1, 5, 1],
    "Shrek":      [2, 2, 4, 2],
    "Arrival":    [np.nan, 5, 1, 5],
    "Mamma Mia":  [1, 1, 5, np.nan]
}, index=[
    "Karen",
    "Bob",
    "Sam",
    "Chloe"
])

ratings

,Matrix,Dune,Barbie,Shrek,Arrival,Mamma Mia
Karen,5,5,NaN,2,NaN,1.0
Bob,5,4,1.0,2,5.0,1.0
Sam,1,1,5.0,4,1.0,5.0
Chloe,4,5,1.0,2,5.0,NaN


## Comparing explicit ratings

For the 1–5 star ratings in this notebook, we use a deliberately simple comparison:

> **Average absolute rating difference on items both users/items have ratings for.**

For example, if two users differ by `0`, `1`, `3`, and `0` stars on four common movies, their average difference is `(0 + 1 + 3 + 0) / 4 = 1.0`.

**Smaller difference = more similar.**

This measure is chosen for transparency in this introductory notebook. It is not the only possible similarity measure.

In [6]:
def average_rating_difference(a, b):
    """
    Return the mean absolute difference using only positions
    observed in BOTH Series. Lower means more similar.
    Q: Why don't we replace missing values with 0? What would happen if we did that?
    """
    common = a.notna() & b.notna()

    x = a[common].to_numpy(dtype=float)
    y = b[common].to_numpy(dtype=float)

    if len(x) == 0:
        return np.nan

    return np.mean(np.abs(x - y))

# 3. User-based collaborative filtering

For user-based collaborative filtering:

> Find users who gave **similar ratings** to the target user.

We compare **rows** of the ratings matrix.

First compare **Karen** and **Bob** using only movies they both rated.

In [7]:
karen_bob_common = pd.concat(
    [ratings.loc["Karen"], ratings.loc["Bob"]],
    axis=1
).dropna()
karen_bob_common.columns = ["Karen", "Bob"]

karen_bob_common

,Karen,Bob
Matrix,5.0,5.0
Dune,5.0,4.0
Shrek,2.0,2.0
Mamma Mia,1.0,1.0


In [8]:
karen_bob_difference = average_rating_difference(
    ratings.loc["Karen"],
    ratings.loc["Bob"]
)

karen_bob_difference

np.float64(0.25)

Now calculate Karen's average rating difference from **every other user**.

Remember: **lower = more similar**.

In [9]:
user_differences = {
    user: average_rating_difference(
        ratings.loc["Karen"],
        ratings.loc[user]
    )
    for user in ratings.index
    if user != "Karen"
}

pd.Series(user_differences).sort_values()

Bob      0.250
Chloe    0.333
Sam      3.500
dtype: float64

### Predict Karen's rating for an unseen movie

A simple **k-nearest-neighbours** approach:

1. Keep users who rated the target movie.
2. Find which of those users are most similar to Karen.
3. Take the `k` nearest neighbours.
4. Average their ratings for the target movie.

For this teaching example we use `k = 2`.

In [10]:
def predict_user_based(ratings, target_user, target_item, k=2):
    candidates = ratings.index[
        (ratings.index != target_user) & ratings[target_item].notna()
    ]

    differences = pd.Series({
        user: average_rating_difference(
            ratings.loc[target_user],
            ratings.loc[user]
        )
        for user in candidates
    }).dropna()

    nearest = differences.nsmallest(k).index

    return ratings.loc[nearest, target_item].mean()

In [11]:
arrival_prediction_user = predict_user_based(ratings, "Karen", "Arrival", k=2)
barbie_prediction_user = predict_user_based(ratings, "Karen", "Barbie", k=2)

print("Predicted Karen rating for Arrival:", arrival_prediction_user)
print("Predicted Karen rating for Barbie: ", barbie_prediction_user)

Predicted Karen rating for Arrival: 5.0
Predicted Karen rating for Barbie:  1.0


### Key idea

**User-based CF:** compare **rows**.

> Who rated movies similarly to Karen, and what did those users think of movies Karen has not rated yet?

# 4. Item-based collaborative filtering

For item-based collaborative filtering:

> Find items that receive **similar ratings from users**.

We now compare **columns** instead of rows. The same average-rating-difference function can be reused.

Start with **Dune**. Which movies were rated most similarly to Dune?

In [12]:
item_differences_from_dune = {
    movie: average_rating_difference(
        ratings["Dune"],
        ratings[movie]
    )
    for movie in ratings.columns
    if movie != "Dune"
}

pd.Series(item_differences_from_dune).sort_values()

Arrival      0.333
Matrix       0.500
Shrek        2.750
Barbie       3.667
Mamma Mia    3.667
dtype: float64

For this dataset, **Arrival** and **Matrix** are treated most similarly to Dune.

Important distinction:

- **Content-based:** Dune is similar to Arrival because their **features** are similar.
- **Item-based CF:** Dune is similar to Arrival because **users rated them similarly**.

## Predict an unseen rating with item-based CF

To predict Karen's rating for an unseen movie:

1. Compare the candidate movie with movies Karen already rated.
2. Find the `k` most similar rated movies (smallest average difference).
3. Average Karen's own ratings for those neighbouring movies.

Again, we use `k = 2` for this small teaching example.

In [13]:
def predict_item_based(ratings, target_user, target_item, k=2):
    user_ratings = ratings.loc[target_user]
    rated_items = user_ratings.dropna().index

    differences = pd.Series({
        item: average_rating_difference(
            ratings[target_item],
            ratings[item]
        )
        for item in rated_items
        if item != target_item
    }).dropna()

    nearest_items = differences.nsmallest(k).index

    return user_ratings.loc[nearest_items].mean()

In [23]:
arrival_prediction_item = predict_item_based(ratings, "Karen", "Arrival", k=2)
barbie_prediction_item = predict_item_based(ratings, "Karen", "Barbie", k=2)

print("Predicted Karen rating for Arrival:", arrival_prediction_item)
print("Predicted Karen rating for Barbie: ", barbie_prediction_item)

Predicted Karen rating for Arrival: 5.0
Predicted Karen rating for Barbie:  1.5


### Inspect the Barbie prediction

For Barbie, the nearest items among Karen's already-rated movies should be **Mamma Mia** and **Shrek**.

Karen rated them `1` and `2`, so the simple item-based prediction is `(1 + 2) / 2 = 1.5`.

In [15]:
barbie_item_differences = pd.Series({
    item: average_rating_difference(ratings["Barbie"], ratings[item])
    for item in ratings.loc["Karen"].dropna().index
}).sort_values()

barbie_item_differences

Mamma Mia    0.000
Shrek        1.000
Dune         3.667
Matrix       3.667
dtype: float64

### Important limitation

With tiny data, a low difference may be based on only a few common ratings. For example, Barbie and Mamma Mia are compared using only the users who rated **both** movies.

Real recommender systems normally need more safeguards than this notebook uses. Here we deliberately keep the calculation simple so the collaborative-filtering logic remains visible.

# 5. Precision@K and Recall@K

Suppose the relevant movies for a user are `B`, `D`, and `F`, and the recommender returns:

`B, A, D, C, F`

In [16]:
relevant = {"B", "D", "F"}
recommendations = ["B", "A", "D", "C", "F"]

In [17]:
def precision_at_k(recommendations, relevant, k):
    top_k = recommendations[:k]
    hits = sum(item in relevant for item in top_k)
    return hits / k


def recall_at_k(recommendations, relevant, k):
    top_k = recommendations[:k]
    hits = sum(item in relevant for item in top_k)
    return hits / len(relevant)

In [18]:
print("Precision@5:", precision_at_k(recommendations, relevant, 5))
print("Recall@5:   ", recall_at_k(recommendations, relevant, 5))

Precision@5: 0.6
Recall@5:    1.0


Interpretation:

- **Precision@5 = 0.6** → 3 of the 5 recommended items were relevant.
- **Recall@5 = 1.0** → all 3 relevant items were retrieved.

But Precision@K and Recall@K do not distinguish well between rankings that contain the same relevant items at different positions.

# 6. Average Precision

Compare two rankings:

**Model A:** `B, A, D, C, F`  
**Model B:** `A, C, B, D, F`

Both retrieve the same three relevant items, but Model A places relevant items earlier.

In [19]:
model_a = ["B", "A", "D", "C", "F"]
model_b = ["A", "C", "B", "D", "F"]
relevant = {"B", "D", "F"}

In [20]:
def average_precision(recommendations, relevant):
    hits = 0
    precisions = []

    for rank, item in enumerate(recommendations, start=1):
        if item in relevant:
            hits += 1
            precisions.append(hits / rank)

    return sum(precisions) / len(relevant)

In [21]:
print("AP Model A:", average_precision(model_a, relevant))
print("AP Model B:", average_precision(model_b, relevant))

AP Model A: 0.7555555555555555
AP Model B: 0.4777777777777778


Expected interpretation:

- Model A ≈ **0.756**
- Model B ≈ **0.478**

Average Precision rewards rankings where relevant items appear **earlier**.

# 7. Summary

| Concept | What is compared? | Method used here |
|---|---|---|
| Content-based | User profile vs. item features | Cosine similarity |
| User-based CF | User rating rows | Average absolute rating difference |
| Item-based CF | Item rating columns | Average absolute rating difference |
| Precision@K | Top-K recommendations vs. relevant items | Hit fraction in Top-K |
| Recall@K | Retrieved relevant items vs. all relevant items | Coverage of relevant set |
| Average Precision | Positions of relevant items in the ranking | Precision at each hit |

## Final reflection

For each recommender system, ask:

1. What data are we using?
2. What exactly are we comparing?
3. How are we deciding what is similar?
4. How do we produce a predicted score or ranking?
5. How do we evaluate whether that ranking is useful?